In [1]:
import pandas as pd
import numpy as np

In [2]:
class SoftmaxRegression:
    def __init__(self, lr=0.01, epochs=5000):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.bias = None

    def __cross_entropy_loss__(self, y_true, y_pred):
        n_samples = y_true.shape[0]
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        return (-1/n_samples) * np.sum(y_true * np.log(y_pred))

    def __softmax__(self, pred):
        shift_pred = pred - np.max(pred, axis = 1, keepdims = True)
        exp_pred = np.exp(shift_pred)
        return exp_pred / np.sum(exp_pred, axis = 1, keepdims = True)

    def __one_hot_encode__(self, y):
        n_classes = len(np.unique(y))
        one_hot = np.zeros((len(y), n_classes))
        one_hot[np.arange(len(y)), y] = 1
        return one_hot

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)

        n_samples, n_features = X.shape
        k_classes = len(np.unique(y))
        self.weights = np.zeros((n_features, k_classes))
        self.bias = np.zeros((1, k_classes))

        y_encoded = self.__one_hot_encode__(y)

        for i in range(self.epochs):
            y_pred = np.dot(X, self.weights) + self.bias
            y_pred = self.__softmax__(y_pred)

            err = y_pred - y_encoded

            dw = (1 / n_samples) * np.dot(X.T, err)
            db = (1 / n_samples) * np.sum(err, axis = 0, keepdims = True)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            if i % 100 == 0:
                loss = self.__cross_entropy_loss__(y_encoded, y_pred)
                # print(f"Epoch {i:4d} | Loss: {loss:.4f}")

    def predict(self, X):
        pred = np.dot(X, self.weights) + self.bias
        y_pred = self.__softmax__(pred)
        return np.argmax(y_pred, axis = 1)


In [4]:
# 1. Create a simple toy dataset (3 classes, 2 features)
np.random.seed(42)

# Class 0: Centered around (2, 2)
X1 = np.random.randn(50, 2) + np.array([2, 2])
y1 = np.zeros(50, dtype=int)

# Class 1: Centered around (-2, -2)
X2 = np.random.randn(50, 2) + np.array([-2, -2])
y2 = np.ones(50, dtype=int)

# Class 2: Centered around (2, -2)
X3 = np.random.randn(50, 2) + np.array([2, -2])
y3 = np.full(50, 2, dtype=int)

# Combine data
X = np.vstack([X1, X2, X3])
y = np.concatenate([y1, y2, y3])

# 2. Initialize and train the model
print("Training Softmax Regression...")
model = SoftmaxRegression(lr=0.01, epochs=5000)
model.fit(X, y)

# 3. Make predictions and check accuracy
predictions = model.predict(X)
accuracy = np.mean(predictions == y) * 100
print(predictions == y)
print(f"\nFinal Training Accuracy: {accuracy:.2f}%")

Training Softmax Regression...
[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True False  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True False  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True False
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True]

Final Training Accuracy: 98.00%
